In [1]:
import ast
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
df = pd.read_csv('../data/raw/wiki_movie_plots_deduped.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34886 entries, 0 to 34885
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Release Year      34886 non-null  int64
 1   Title             34886 non-null  str  
 2   Origin/Ethnicity  34886 non-null  str  
 3   Director          34886 non-null  str  
 4   Cast              33464 non-null  str  
 5   Genre             34886 non-null  str  
 6   Wiki Page         34886 non-null  str  
 7   Plot              34886 non-null  str  
dtypes: int64(1), str(7)
memory usage: 78.9 MB


In [4]:
OriginsList = ['American','Canadian','Australian','Egyptian','British']
df['Origin/Ethnicity'].unique()

<ArrowStringArray>
[    'American',   'Australian',  'Bangladeshi',      'British',
     'Canadian',      'Chinese',     'Egyptian',    'Hong Kong',
     'Filipino',     'Assamese',      'Bengali',    'Bollywood',
      'Kannada',    'Malayalam',      'Marathi',      'Punjabi',
        'Tamil',       'Telugu',     'Japanese',    'Malaysian',
    'Maldivian',      'Russian', 'South_Korean',      'Turkish']
Length: 24, dtype: str

In [5]:
df.tail()

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
34881,2014,The Water Diviner,Turkish,Director: Russell Crowe,Director: Russell Crowe\r\nCast: Russell Crowe...,unknown,https://en.wikipedia.org/wiki/The_Water_Diviner,"The film begins in 1919, just after World War ..."
34882,2017,Çalgı Çengi İkimiz,Turkish,Selçuk Aydemir,"Ahmet Kural, Murat Cemcir",comedy,https://en.wikipedia.org/wiki/%C3%87alg%C4%B1_...,"Two musicians, Salih and Gürkan, described the..."
34883,2017,Olanlar Oldu,Turkish,Hakan Algül,"Ata Demirer, Tuvana Türkay, Ülkü Duru",comedy,https://en.wikipedia.org/wiki/Olanlar_Oldu,"Zafer, a sailor living with his mother Döndü i..."
34884,2017,Non-Transferable,Turkish,Brendan Bradley,"YouTubers Shanna Malcolm, Shira Lazar, Sara Fl...",romantic comedy,https://en.wikipedia.org/wiki/Non-Transferable...,The film centres around a young woman named Am...
34885,2017,İstanbul Kırmızısı,Turkish,Ferzan Özpetek,"Halit Ergenç, Tuba Büyüküstün, Mehmet Günsür, ...",romantic,https://en.wikipedia.org/wiki/%C4%B0stanbul_K%...,The writer Orhan Şahin returns to İstanbul aft...


In [6]:
print((df['Genre'].str.lower() == 'unknown').sum())

6083


In [7]:
df = df.dropna(subset=['Plot', 'Genre'])
df = df[df['Genre'].str.lower() != 'unknown']
df.isna().sum()

Release Year          0
Title                 0
Origin/Ethnicity      0
Director              0
Cast                739
Genre                 0
Wiki Page             0
Plot                  0
dtype: int64

In [8]:
df.loc[(df['Origin/Ethnicity'].isin(['Egyptian']))]

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
22896,1940,Yawm Said (Happy Day),Egyptian,Mohammed Karim,"Mohamed Abd El Wahab, Faten Hamama",drama,https://en.wikipedia.org/wiki/Yawm_Said,Abdel Wahab plays the role of a young man who ...
22897,1944,Rossassa Fel Qalb (Bullet in the Heart),Egyptian,Mohammed Karim,"Mohammed Abdel Wahab, Raqiya Ibrahim, Faten Ha...",drama,https://en.wikipedia.org/wiki/Rossassa_Fel_Qalb,Mohsen is a ladies' man. He has a close friend...
22898,1946,Malak al-Rahma (The Angel of Mercy),Egyptian,Youssef Wahbi,"Faten Hamama, Farid Shawki, Youssef Wahbi",drama,https://en.wikipedia.org/wiki/The_Angel_of_Mer...,Fouad Bek is married to Imtethal and has a dau...
22899,1947,Abu Zayd al-Hilali,Egyptian,Ezzel Dine Zulficar,"Seraj Munir, Faten Hamama",biography,https://en.wikipedia.org/wiki/Abu_Zayd_al-Hila...,Abu Zayd al-Hilali's son and wife escape and y...
22900,1948,Khulood (Immortality),Egyptian,Ezzel Dine Zulficar,"Kamal al-Shennawi, Faten Hamama, Ismail Yasseen",romance,https://en.wikipedia.org/wiki/Khulood,"A man named Mahmoud falls in love with Layla, ..."
...,...,...,...,...,...,...,...,...
22958,1992,Al-Irhab Wal Kabab (Terrorism and Kebab),Egyptian,Sherif Arafa,"Adel Emam, Yousra, Kamal el-Shennawi",political comedy,https://en.wikipedia.org/wiki/Terrorism_and_Kebab,The action primarily takes place in The Mogamm...
22959,1994,Al-Irhabi (The Terrorist),Egyptian,Nader Galal,"Adel Emam, Sherine, Madiha Yousri, Salah Zulfikar",political / drama,https://en.wikipedia.org/wiki/The_Terrorist_(1...,"Adel Imam plays Brother Ali, an Islamic radica..."
22960,1994,Al-Mohager (The Emigrant),Egyptian,Youssef Chahine,"Khaled El Nabawy, Hanan Tork, Youssra",drama,https://en.wikipedia.org/wiki/Al-Mohager,"In this film, young Ram is a thinker who has g..."
22961,1999,El Akhar (The Other),Egyptian,Youssef Chahine,"Nabila Ebeid, Mahmoud Hemida, Hanan Tork, Hani...",drama,https://en.wikipedia.org/wiki/The_Other_(1999_...,"Love sparks when Adam, back visiting Cairo fro..."


In [9]:
df.info()

<class 'pandas.DataFrame'>
Index: 28803 entries, 6 to 34885
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Release Year      28803 non-null  int64
 1   Title             28803 non-null  str  
 2   Origin/Ethnicity  28803 non-null  str  
 3   Director          28803 non-null  str  
 4   Cast              28064 non-null  str  
 5   Genre             28803 non-null  str  
 6   Wiki Page         28803 non-null  str  
 7   Plot              28803 non-null  str  
dtypes: int64(1), str(7)
memory usage: 67.5 MB


In [10]:
df = df[df['Origin/Ethnicity'].isin(OriginsList)]

In [11]:
print(list(df['Origin/Ethnicity'].unique()))
df.info()

['American', 'Australian', 'British', 'Canadian', 'Egyptian']
<class 'pandas.DataFrame'>
Index: 21259 entries, 6 to 22962
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Release Year      21259 non-null  int64
 1   Title             21259 non-null  str  
 2   Origin/Ethnicity  21259 non-null  str  
 3   Director          21259 non-null  str  
 4   Cast              20891 non-null  str  
 5   Genre             21259 non-null  str  
 6   Wiki Page         21259 non-null  str  
 7   Plot              21259 non-null  str  
dtypes: int64(1), str(7)
memory usage: 51.4 MB


In [12]:
list(df['Genre'].unique())

['western',
 'comedy',
 'short',
 'short action/crime western',
 'short film',
 'biographical',
 'drama',
 'adventure',
 'short fantasy',
 'silent sports',
 'horror',
 'crime',
 'drama, horror',
 'historical drama',
 'fantasy drama',
 'biographical drama',
 'documentary drama',
 'fantasy',
 'adventure serial',
 'epic',
 'historical',
 'comedy short',
 'comedy, western',
 'biography',
 'action adventure',
 'western drama',
 'short comedy',
 'comedy–drama',
 'romantic drama',
 'mystery',
 'crime drama',
 'romance',
 'sexual hygiene/exploitation film',
 'comedy drama',
 'war drama',
 'spy',
 'romantic comedy',
 'propaganda',
 'ww1 propaganda',
 'biopic',
 'animated series',
 'drama romance',
 'melodrama',
 'period drama',
 'swashbuckler',
 'romance drama',
 'drama, adventure',
 'crime comedy',
 'documentary',
 'comedy western',
 'fantasy, family',
 'war',
 'comedy, adventure',
 'fantasy, adventure',
 'thriller',
 'dramatic comedy',
 'romantic comedy/drama',
 'mystery, thriller',
 'crime t

In [13]:
GENRE_MAP = {
    'science fiction': 'sci-fi',
    'romantic comedy': 'rom-com',
    'romantic': 'romance',
    'biographical': 'biography',
    'biopic': 'biography',
}


def cleanGenre(g):
    g = str(g).lower()
    for key, val in GENRE_MAP.items():
        g = g.replace(key, val)
    g = re.sub(r'\b(film|short|series|serial|movie)\b', '', g)

    
    genres = re.split(r'[/,&\s]+', g)
    return [x for x in genres if len(x) > 2]


df['GenreList'] = df['Genre'].apply(cleanGenre)

allGenres = [g for sub in df['GenreList'] for g in sub]
Top15 = pd.Series(allGenres).value_counts().head(20).index.tolist()

df['GenreList'] = df['GenreList'].apply(
    lambda list_: [g for g in list_ if g in Top15]
)
df = df[df['GenreList'].map(len) > 0]
print(Top15)

['drama', 'comedy', 'crime', 'horror', 'thriller', 'western', 'action', 'musical', 'sci-fi', 'adventure', 'war', 'romance', 'animated', 'mystery', 'biography', 'family', 'rom-com', 'noir', 'fantasy', 'animation']


In [14]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def lemmatize_plot(text):
    clean_text = re.sub(r'[^a-z\s]', '', str(text).lower())
    words = clean_text.split()

    clean_words = [
        lemmatizer.lemmatize(lemmatizer.lemmatize(w, pos='v'), pos='n')
        for w in words
        if w not in stop_words
    ]
    return ' '.join(clean_words)


df['CleanPlot'] = df['Plot'].apply(lemmatize_plot)

In [15]:
df.info()

<class 'pandas.DataFrame'>
Index: 20233 entries, 6 to 22962
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Release Year      20233 non-null  int64 
 1   Title             20233 non-null  str   
 2   Origin/Ethnicity  20233 non-null  str   
 3   Director          20233 non-null  str   
 4   Cast              19920 non-null  str   
 5   Genre             20233 non-null  str   
 6   Wiki Page         20233 non-null  str   
 7   Plot              20233 non-null  str   
 8   GenreList         20233 non-null  object
 9   CleanPlot         20233 non-null  str   
dtypes: int64(1), object(1), str(8)
memory usage: 77.7+ MB


In [16]:
df.head()

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot,GenreList,CleanPlot
6,1903,The Great Train Robbery,American,Edwin S. Porter,NaN,western,https://en.wikipedia.org/wiki/The_Great_Train_...,The film opens with two bandits breaking into ...,[western],film open two bandit break railroad telegraph ...
7,1904,The Suburbanite,American,Wallace McCutcheon,NaN,comedy,https://en.wikipedia.org/wiki/The_Suburbanite,The film is about a family who move to the sub...,[comedy],film family move suburb hop quiet life thing s...
11,1906,From Leadville to Aspen: A Hold-Up in the Rockies,American,Francis J. Marion and Wallace McCutcheon,NaN,short action/crime western,https://en.wikipedia.org/wiki/From_Leadville_t...,The film features a train traveling through th...,"[action, crime, western]",film feature train travel rockies hold create ...
13,1907,Daniel Boone,American,Wallace McCutcheon and Ediwin S. Porter,"William Craven, Florence Lawrence",biographical,https://en.wikipedia.org/wiki/Daniel_Boone_(19...,Boone's daughter befriends an Indian maiden as...,[biography],boone daughter befriend indian maiden boone co...
14,1907,How Brown Saw the Baseball Game,American,Unknown,Unknown,comedy,https://en.wikipedia.org/wiki/How_Brown_Saw_th...,Before heading out to a baseball game at a nea...,[comedy],head baseball game nearby ballpark sport fan m...


In [17]:

df[['Title', 'Release Year', 'Director', 'GenreList', 'Plot', 'CleanPlot']].to_parquet(
    '../data/processed/plotify_cleaned_movies.parquet', index=False
)